# Import required packages

In [14]:
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
import joblib


## Load csv

In [11]:
df = pd.read_csv("cleaned_dataset.csv", sep=";")

# Prepare the training

In [17]:
y = df["Retard moyen de tous les trains à l'arrivée"]
X = df.drop(columns=["Retard moyen de tous les trains à l'arrivée"])
categorical_cols = ["Service", "Gare de départ", "Gare d'arrivée"]
cols_to_remove = [col for col in X.columns if "Retard" in col or "retard" in col]
X = X.drop(columns=cols_to_remove)

numeric_cols = [col for col in X.columns if col not in categorical_cols]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

print(numeric_cols)

['Durée moyenne du trajet', 'Nombre de circulations prévues', 'Nombre de trains annulés', 'Month', 'Year']


## Cross-Validation

In [18]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = -cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="neg_mean_absolute_error"
)

rmse_scores = (-cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="neg_root_mean_squared_error"
))

print("MAE CV:", mae_scores.mean())
print("RMSE CV:", rmse_scores.mean())
print("MAE std:", mae_scores.std())

MAE CV: 2.419628813824954
RMSE CV: 3.618885654164598
MAE std: 0.0438739694026826


# Train the model

In [19]:
model.fit(X, y)
print("Model trained")


Model trained


# Evaluate

In [20]:
y_pred = model.predict(X)
mae_total = mean_absolute_error(y, y_pred)
print("MAE sur le dataset :", mae_total)


MAE sur le dataset : 2.3677007914641894


# Save model file

In [21]:
joblib.dump(model, "model.pkl")
print("Model saved in 'model.pkl")

Model saved in 'model.pkl
